# Zomato Bangalore Restaurant Analytics
## Notebook 3: Visualizations & Insights

Build charts and visualizations around customer segments and online ordering patterns.

---


In [ ]:
# Imports and Style Setup
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
import re
import warnings
warnings.filterwarnings('ignore')

# Dark professional theme
DARK_BG = '#0d1117'
CARD_BG = '#161b22'
TEXT_COLOR = '#e6edf3'
GRID_COLOR = '#21262d'
ACCENT_1 = '#58a6ff'
ACCENT_2 = '#f78166'

plt.rcParams.update({
    'figure.facecolor': DARK_BG,
    'axes.facecolor': CARD_BG,
    'axes.edgecolor': GRID_COLOR,
    'axes.labelcolor': TEXT_COLOR,
    'text.color': TEXT_COLOR,
    'xtick.color': TEXT_COLOR,
    'ytick.color': TEXT_COLOR,
    'grid.color': GRID_COLOR,
    'grid.alpha': 0.3,
    'figure.figsize': (14, 7),
    'font.size': 12,
    'axes.titlesize': 18,
    'axes.labelsize': 14,
})

print('Libraries loaded with dark theme.')

In [ ]:
# Connect to Database
conn = sqlite3.connect('../zomato_bangalore.db')

def fetch_data(query):
    return pd.read_sql_query(query, conn)

print('Connected to database.')

---
## 1. Online Orders
Looking at whether online ordering correlates with higher ratings.

In [ ]:
# Online vs Offline Rating Distribution
df_online = fetch_data("""
    SELECT rating, online_order
    FROM restaurants
    WHERE rating IS NOT NULL
""")
df_online['Order Type'] = df_online['online_order'].map({1: 'Online Orders', 0: 'No Online Orders'})

fig, ax = plt.subplots(figsize=(14, 7))
for order_type, color, alpha in [('Online Orders', ACCENT_1, 0.4), ('No Online Orders', ACCENT_2, 0.3)]:
    subset = df_online[df_online['Order Type'] == order_type]['rating']
    sns.kdeplot(subset, ax=ax, fill=True, alpha=alpha, color=color, label=order_type, linewidth=2.5, bw_adjust=0.8)

avg_on = df_online[df_online['online_order']==1]['rating'].mean()
avg_off = df_online[df_online['online_order']==0]['rating'].mean()
ax.axvline(avg_on, color=ACCENT_1, linestyle='--', linewidth=2, alpha=0.8)
ax.axvline(avg_off, color=ACCENT_2, linestyle='--', linewidth=2, alpha=0.8)
ax.annotate(f'Avg: {avg_on:.2f}', xy=(avg_on, 0.95), xycoords=('data', 'axes fraction'),
            fontsize=12, color=ACCENT_1, fontweight='bold', ha='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=CARD_BG, edgecolor=ACCENT_1, alpha=0.9))
ax.annotate(f'Avg: {avg_off:.2f}', xy=(avg_off, 0.85), xycoords=('data', 'axes fraction'),
            fontsize=12, color=ACCENT_2, fontweight='bold', ha='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=CARD_BG, edgecolor=ACCENT_2, alpha=0.9))

ax.set_title('Rating Distribution: Online vs Offline Orders', fontsize=20, fontweight='bold', pad=20)
ax.set_xlabel('Rating (out of 5)')
ax.set_ylabel('Density')
ax.legend(fontsize=13, framealpha=0.3, edgecolor=GRID_COLOR, facecolor=CARD_BG)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../images/online_vs_offline_rating.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

> Restaurants with online ordering tend to rate higher (clustered around 3.8-4.0) compared to those without it.

---
## 2. Customer Segments (Budget vs Luxury)

In [ ]:
# Customer Segmentation Treemap
df_segments = fetch_data("""
    WITH Segmented AS (
        SELECT location, approx_cost, rating,
            NTILE(4) OVER (ORDER BY approx_cost) as quartile
        FROM restaurants
        WHERE approx_cost IS NOT NULL AND rating IS NOT NULL
    )
    SELECT location,
        CASE quartile WHEN 1 THEN 'Budget' WHEN 2 THEN 'Mid-Range'
             WHEN 3 THEN 'Premium' WHEN 4 THEN 'Luxury' END as Segment,
        AVG(rating) as avg_rating,
        COUNT(*) as total_restaurants
    FROM Segmented
    GROUP BY location, quartile
    HAVING total_restaurants > 20
""")

fig = px.treemap(df_segments,
    path=[px.Constant('Bangalore'), 'Segment', 'location'],
    values='total_restaurants', color='avg_rating',
    color_continuous_scale='RdYlGn',
    title='Restaurant Market Segmentation by Location and Rating')
fig.update_layout(margin=dict(t=60, l=25, r=25, b=25),
    paper_bgcolor=DARK_BG, plot_bgcolor=CARD_BG,
    font=dict(color=TEXT_COLOR, size=13), title_font=dict(size=20),
    coloraxis_colorbar=dict(title=dict(text='Avg Rating', font=dict(color=TEXT_COLOR)),
                            tickfont=dict(color=TEXT_COLOR)))
fig.write_html('../images/segmentation_treemap.html')
fig.show()

> The premium and luxury segments rate higher on average, but budget restaurants make up most of the market — especially in BTM and HSR.

---
## 3. Market Saturation

In [ ]:
# Market Saturation Scatter
df_loc = fetch_data("""
    SELECT location, COUNT(*) as restaurant_count,
        AVG(rating) as avg_rating, AVG(approx_cost) as avg_cost
    FROM restaurants
    WHERE location IS NOT NULL AND rating IS NOT NULL
    GROUP BY location
    HAVING restaurant_count > 50
""")

fig = px.scatter(df_loc, x='restaurant_count', y='avg_rating',
    size='avg_cost', color='avg_rating', hover_name='location',
    title='Market Saturation: Restaurant Density vs Average Rating',
    labels={'restaurant_count': 'Number of Restaurants', 'avg_rating': 'Average Rating'},
    color_continuous_scale='Turbo', size_max=30, text='location')
fig.update_traces(textposition='top center', textfont=dict(size=10, color=TEXT_COLOR))
fig.update_layout(paper_bgcolor=DARK_BG, plot_bgcolor=CARD_BG,
    font=dict(color=TEXT_COLOR, size=13), title_font=dict(size=20),
    xaxis=dict(gridcolor=GRID_COLOR, zeroline=False),
    yaxis=dict(gridcolor=GRID_COLOR, zeroline=False),
    coloraxis_colorbar=dict(title=dict(text='Avg Rating', font=dict(color=TEXT_COLOR)),
                            tickfont=dict(color=TEXT_COLOR)),
    width=1200, height=700)
fig.add_hline(y=df_loc['avg_rating'].mean(), line_dash='dot', line_color='#8b949e',
              annotation_text='City Avg Rating', annotation_font_color=TEXT_COLOR)
fig.add_vline(x=df_loc['restaurant_count'].mean(), line_dash='dot', line_color='#8b949e',
              annotation_text='Avg Density', annotation_font_color=TEXT_COLOR)
fig.write_html('../images/market_saturation.html')
fig.show()

> Areas in the top-left (few restaurants, high ratings) look like good opportunities. Areas in the bottom-right (lots of restaurants, lower ratings) are already crowded.

---
## 4. Popular Dishes

In [ ]:
# Word Cloud — with data cleanup to remove garbage entries
df_dishes = fetch_data('SELECT dish FROM restaurant_dishes')

def clean_dish(d):
    if pd.isna(d): return ''
    d = str(d)
    bad_words = ['rated', 'rating', 'review', 'visited', 'ordered', '\\n',
                 'restaurant', 'ambience', 'ambiance', 'staff', 'service',
                 'place', 'experience', 'decor', 'music', 'drink',
                 'avoid', 'kindly', 'company', 'x83', 'x98', 'x9f', 'xa0']
    if any(w in d.lower() for w in bad_words): return ''
    if d.startswith(('(', '"', "'")): return ''
    if len(d) < 2 or len(d) > 50: return ''
    if not re.match(r'^[A-Za-z\s\-\'\. ]+$', d): return ''
    return d.strip()

clean_dishes = df_dishes['dish'].apply(clean_dish)
text = ' '.join(clean_dishes[clean_dishes != ''].values)

wordcloud = WordCloud(width=1600, height=800, background_color=DARK_BG,
    colormap='cool', max_words=150, max_font_size=120, min_font_size=12,
    prefer_horizontal=0.7, relative_scaling=0.5, margin=20).generate(text)

fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(wordcloud, interpolation='bilinear')
ax.axis('off')
ax.set_title('Popular Dishes in Bangalore', fontsize=22, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../images/dishes_wordcloud.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

In [ ]:
conn.close()
print('All visualizations generated and saved to /images folder.')